# LeetCode 3416 — Subsequences with a Unique Middle Mode II bằng Python

- Link: https://leetcode.com/problems/subsequences-with-a-unique-middle-mode-ii/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **Tổ hợp đếm**, cố định **phần tử giữa**, chia trường hợp theo số lần xuất hiện của giá trị giữa, **duy trì các tổng Σ cl·cr, Σ cl·cr², …** khi tâm di chuyển

> **Ghi chú Premium:** bài này là Premium nên LeetCode không trả về đề qua API công khai. Đề và ví dụ lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3416/). Trang doocs không hiển thị mã nên **tên hàm `subsequencesWithMiddleMode(self, nums)` được suy ra** từ bản I (LC 3395, cùng đề nhưng `n ≤ 1000`). Vui lòng **đối chiếu lại** chữ ký nếu có tài khoản Premium.

Skeleton (suy ra từ bản I):

```python
class Solution:
    def subsequencesWithMiddleMode(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Cố định **chỉ số giữa** `i` (giá trị `x`) của dãy con độ dài 5: chọn 2 chỉ số bên trái, 2 bên phải.
2. Chia trường hợp theo tổng số lần `x` xuất hiện trong dãy con: **≥ 3** (luôn là mode duy nhất) và **đúng 2** (ba phần tử còn lại phải **đôi một khác nhau** và khác `x`).
3. Đếm trường hợp "đúng 2" bằng công thức qua các tổng `Σ_v cl[v]·cr[v]`, `Σ cl·cr²`, `Σ cl²·cr`, `Σ cl²`, `Σ cr²`.
4. Cập nhật các tổng trong `O(1)` khi chuyển tâm từ `i` sang `i + 1` (chỉ một giá trị thay đổi đếm).
5. Kiểm chứng bằng vét cạn và một bản `O(n · D)` tính lại từ đầu.

## 2. Đề bài & ví dụ

Cho mảng `nums`, đếm số **dãy con độ dài 5** (theo chỉ số) có **mode duy nhất ở giữa**: phần tử thứ 3 (`seq[2]`) xuất hiện **nhiều lần nhất** trong dãy con và **không hòa** với giá trị nào khác. Trả về kết quả mod `10^9 + 7`.

**Ví dụ 1:** `[1,1,1,1,1,1]` → `6`. **Ví dụ 2:** `[1,2,2,3,3,4]` → `4`. **Ví dụ 3:** `[0,1,...,8]` → `0`.

**Ràng buộc (theo doocs):** `5 ≤ n ≤ 10^5`, `−10^9 ≤ nums[i] ≤ 10^9`.

In [1]:
from typing import List
from collections import Counter
import itertools
from math import comb

MOD = 10 ** 9 + 7
examples = [([1, 1, 1, 1, 1, 1], 6), ([1, 2, 2, 3, 3, 4], 4), (list(range(9)), 0)]
for a, exp in examples:
    print(a, '-> kỳ vọng', exp)

[1, 1, 1, 1, 1, 1] -> kỳ vọng 6
[1, 2, 2, 3, 3, 4] -> kỳ vọng 4
[0, 1, 2, 3, 4, 5, 6, 7, 8] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Mode duy nhất ở giữa với dãy 5 phần tử

Gọi `t` = số lần giá trị giữa `x` xuất hiện trong dãy con (tính cả chính nó).

- `t ≥ 3`: các giá trị khác chiếm ≤ 2 vị trí ⇒ xuất hiện ≤ 2 < `t` ⇒ luôn hợp lệ.
- `t = 2`: ba vị trí còn lại phải là **ba giá trị khác nhau** (khác `x`); nếu có giá trị lặp 2 lần thì hòa.
- `t = 1`: các giá trị khác phải xuất hiện < 1 lần — không thể (còn 4 vị trí).

In [2]:
def unique_middle_mode(seq):
    c = Counter(seq)
    m = seq[2]
    return all(c[m] > v for k, v in c.items() if k != m)

print(unique_middle_mode([1, 2, 2, 3, 4]), unique_middle_mode([1, 2, 2, 3, 3]), unique_middle_mode([5, 1, 5, 1, 5]))

True False True


### 3.2. Đếm cặp chỉ số có giá trị khác nhau

Trong một tập có `R` phần tử, số cặp (không thứ tự) có **giá trị khác nhau** `= C(R, 2) − Σ_v C(r_v, 2)`. Số cặp khác giá trị **không chứa** giá trị `v`: trừ thêm `r_v · (R − r_v)`.

In [3]:
vals = [1, 1, 2, 3, 3, 3]
R = len(vals); cnt = Counter(vals)
P = comb(R, 2) - sum(comb(c, 2) for c in cnt.values())
print('cặp khác giá trị:', P, '| đếm trực tiếp:', sum(1 for i, j in itertools.combinations(range(R), 2) if vals[i] != vals[j]))
print('không chứa giá trị 3:', P - cnt[3] * (R - cnt[3]),
      '| trực tiếp:', sum(1 for i, j in itertools.combinations(range(R), 2) if vals[i] != vals[j] and 3 not in (vals[i], vals[j])))

cặp khác giá trị: 11 | đếm trực tiếp: 11
không chứa giá trị 3: 2 | trực tiếp: 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2 `[1,2,2,3,3,4]`: các dãy con hợp lệ là `[1,2,2,3,4]` (giữa là `2` ở chỉ số 2, `t = 2`, ba giá trị còn lại `1, 3, 4` khác nhau) — có 2 cách chọn số `3` (chỉ số 3 hoặc 4) — và `[1,2,3,3,4]` (giữa là `3` ở chỉ số 3, `t = 2`) — 2 cách chọn số `2`. Tổng `4`. Dãy `[1,2,2,3,3]` không hợp lệ vì `2` và `3` hòa.

Ví dụ 1: mọi dãy con là `[1,1,1,1,1]` (`t = 5`), có `C(6, 5) = 6` cách.

In [4]:
a = [1, 2, 2, 3, 3, 4]
good = [tuple(idx) for idx in itertools.combinations(range(6), 5) if unique_middle_mode([a[i] for i in idx])]
print(good, [[a[i] for i in g] for g in good])

[(0, 1, 2, 3, 5), (0, 1, 2, 4, 5), (0, 1, 3, 4, 5), (0, 2, 3, 4, 5)] [[1, 2, 2, 3, 4], [1, 2, 2, 3, 4], [1, 2, 3, 3, 4], [1, 2, 3, 3, 4]]


## 5. Cách 1 — Brute Force: mọi bộ 5 chỉ số

In [5]:
def count_brute(nums):
    return sum(1 for idx in itertools.combinations(range(len(nums)), 5)
               if unique_middle_mode([nums[i] for i in idx])) % MOD

for a, exp in examples:
    print(count_brute(a), '| kỳ vọng', exp)

6 | kỳ vọng 6
4 | kỳ vọng 4
0 | kỳ vọng 0


**Độ phức tạp:** `O(C(n, 5))` — chỉ cho `n ≤ 15`.

## 6. Ý tưởng thuật toán / bước đột phá

Tại tâm `i` (giá trị `x`): bên trái có `a` số bằng `x` và `L' = i − a` số khác `x`; bên phải có `b` và `R' = n − 1 − i − b`. Gọi `lc[v]`, `rc[v]` là số lần giá trị `v ≠ x` ở hai bên.

**(A) `t ≥ 3`:** chọn `j` số `x` bên trái và `k` bên phải với `j + k ≥ 2`:
`Σ_{j,k ∈ {0,1,2}, j+k≥2} C(a, j) C(L', 2−j) C(b, k) C(R', 2−k)`.

**(B) `t = 2`, số `x` thêm nằm bên trái (`j = 1, k = 0`):** chọn 1 số `x` (`a` cách), 1 số `y ≠ x` bên trái, cặp `{z1, z2}` bên phải khác `x`, với `y, z1, z2` đôi một khác giá trị:

`a · ( L' · P − Σ_{v≠x} lc[v] · rc[v] · (R' − rc[v]) )`, với `P = C(R', 2) − Σ_{v≠x} C(rc[v], 2)`.

**(C) Đối xứng** (`j = 0, k = 1`): `b · ( R' · Q − Σ_{v≠x} rc[v] · lc[v] · (L' − lc[v]) )`, `Q = C(L', 2) − Σ_{v≠x} C(lc[v], 2)`.

Các tổng `Σ_{v≠x}` lấy từ tổng toàn cục trừ đi số hạng của `v = x`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Công thức theo tâm, tính lại các bộ đếm mỗi tâm (`O(n · D)`)

In [6]:
def C2(x):
    return x * (x - 1) // 2

def count_per_center(nums):
    n, total = len(nums), 0
    for i in range(n):
        x = nums[i]
        L, R = Counter(nums[:i]), Counter(nums[i + 1:])
        a, b = L[x], R[x]
        Lp, Rp = i - a, n - 1 - i - b
        for j in range(3):
            for k in range(3):
                if j + k >= 2:
                    total += comb(a, j) * comb(Lp, 2 - j) * comb(b, k) * comb(Rp, 2 - k)
        lc = {v: c for v, c in L.items() if v != x}
        rc = {v: c for v, c in R.items() if v != x}
        P = C2(Rp) - sum(C2(c) for c in rc.values())
        Q = C2(Lp) - sum(C2(c) for c in lc.values())
        t1 = Lp * P - sum(lc[v] * rc.get(v, 0) * (Rp - rc.get(v, 0)) for v in lc)
        t2 = Rp * Q - sum(rc[v] * lc.get(v, 0) * (Lp - lc.get(v, 0)) for v in rc)
        total += a * t1 + b * t2
    return total % MOD

for a, exp in examples:
    print(count_per_center(a), '| kỳ vọng', exp)

6 | kỳ vọng 6
4 | kỳ vọng 4
0 | kỳ vọng 0


### 7.2. Cách 3 — Duy trì các tổng khi dịch tâm (lời giải chính, `O(n)`)

Giữ `S1 = Σ cl·cr`, `S2 = Σ cl·cr²`, `S4 = Σ cl²·cr`, `SR = Σ cr²`, `SL = Σ cl²` (trên mọi giá trị). Khi phần tử `x` chuyển từ phải sang tâm rồi sang trái, chỉ số hạng của `x` thay đổi: trừ số hạng cũ, cập nhật đếm, cộng số hạng mới. `Σ_v C(rc[v], 2) = (Σ rc² − Σ rc)/2`.

In [7]:
def count_fast(nums):
    n = len(nums)
    cl, cr = Counter(), Counter(nums)
    S = {'S1': 0, 'S2': 0, 'S4': 0, 'SR': sum(c * c for c in cr.values()), 'SL': 0}

    def apply(v, sign):
        l, r = cl[v], cr[v]
        S['S1'] += sign * l * r; S['S2'] += sign * l * r * r; S['S4'] += sign * l * l * r
        S['SR'] += sign * r * r; S['SL'] += sign * l * l

    total = 0
    for i in range(n):
        x = nums[i]
        apply(x, -1); cr[x] -= 1; apply(x, +1)          # x rời bên phải (thành tâm)
        a, b = cl[x], cr[x]
        Lp, Rp = i - a, n - 1 - i - b
        ca, cb, cL, cR = [1, a, C2(a)], [1, b, C2(b)], [1, Lp, C2(Lp)], [1, Rp, C2(Rp)]
        for j in range(3):
            for k in range(3):
                if j + k >= 2:
                    total += ca[j] * cL[2 - j] * cb[k] * cR[2 - k]
        s1 = S['S1'] - a * b; s2 = S['S2'] - a * b * b; s4 = S['S4'] - a * a * b
        sr = S['SR'] - b * b; sl = S['SL'] - a * a
        P = C2(Rp) - (sr - Rp) // 2
        Q = C2(Lp) - (sl - Lp) // 2
        total += a * (Lp * P - (Rp * s1 - s2)) + b * (Rp * Q - (Lp * s1 - s4))
        apply(x, -1); cl[x] += 1; apply(x, +1)          # x sang bên trái
    return total % MOD

for a, exp in examples:
    print(count_fast(a), '| kỳ vọng', exp)

6 | kỳ vọng 6
4 | kỳ vọng 4
0 | kỳ vọng 0


Chú ý: `Σ_{v≠x} lc[v]·rc[v]·(R' − rc[v]) = R'·Σ_{v≠x} lc·rc − Σ_{v≠x} lc·rc²` — đó là `Rp * s1 − s2` trong code.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2: với mỗi tâm in `a, L', b, R'` và đóng góp từng trường hợp.

In [8]:
def trace(nums):
    n = len(nums)
    total = 0
    for i in range(n):
        x = nums[i]
        L, R = Counter(nums[:i]), Counter(nums[i + 1:])
        a, b = L[x], R[x]; Lp, Rp = i - a, n - 1 - i - b
        A = sum(comb(a, j) * comb(Lp, 2 - j) * comb(b, k) * comb(Rp, 2 - k)
                for j in range(3) for k in range(3) if j + k >= 2)
        lc = {v: c for v, c in L.items() if v != x}; rc = {v: c for v, c in R.items() if v != x}
        P = C2(Rp) - sum(C2(c) for c in rc.values()); Q = C2(Lp) - sum(C2(c) for c in lc.values())
        B = a * (Lp * P - sum(lc[v] * rc.get(v, 0) * (Rp - rc.get(v, 0)) for v in lc))
        Cc = b * (Rp * Q - sum(rc[v] * lc.get(v, 0) * (Lp - lc.get(v, 0)) for v in rc))
        total += A + B + Cc
        print(f"tâm i={i} (x={x}): a={a}, L'={Lp}, b={b}, R'={Rp} | t>=3: {A}, t=2 (trái): {B}, t=2 (phải): {Cc}")
    print('tổng:', total)

trace([1, 2, 2, 3, 3, 4])

tâm i=0 (x=1): a=0, L'=0, b=0, R'=5 | t>=3: 0, t=2 (trái): 0, t=2 (phải): 0
tâm i=1 (x=2): a=0, L'=1, b=1, R'=3 | t>=3: 0, t=2 (trái): 0, t=2 (phải): 0
tâm i=2 (x=2): a=1, L'=1, b=0, R'=3 | t>=3: 0, t=2 (trái): 2, t=2 (phải): 0
tâm i=3 (x=3): a=0, L'=3, b=1, R'=1 | t>=3: 0, t=2 (trái): 0, t=2 (phải): 2
tâm i=4 (x=3): a=1, L'=3, b=0, R'=1 | t>=3: 0, t=2 (trái): 0, t=2 (phải): 0
tâm i=5 (x=4): a=0, L'=5, b=0, R'=0 | t>=3: 0, t=2 (trái): 0, t=2 (phải): 0
tổng: 4


## 9. Code nộp LeetCode

(Chữ ký suy ra từ bản I — xem ghi chú Premium ở đầu.)

In [9]:
class Solution:
    def subsequencesWithMiddleMode(self, nums: List[int]) -> int:
        MOD = 10 ** 9 + 7
        n = len(nums)
        cl = {}
        cr = {}
        for v in nums:
            cr[v] = cr.get(v, 0) + 1
        S1 = S2 = S4 = SL = 0
        SR = sum(c * c for c in cr.values())

        def c2(x):
            return x * (x - 1) // 2

        total = 0
        for i in range(n):
            x = nums[i]
            l, r = cl.get(x, 0), cr[x]
            # x rời bên phải
            S1 -= l * r; S2 -= l * r * r; S4 -= l * l * r; SR -= r * r
            r -= 1
            S1 += l * r; S2 += l * r * r; S4 += l * l * r; SR += r * r
            cr[x] = r
            a, b = l, r
            Lp, Rp = i - a, n - 1 - i - b
            ca, cb, cL, cR = (1, a, c2(a)), (1, b, c2(b)), (1, Lp, c2(Lp)), (1, Rp, c2(Rp))
            for j in range(3):
                for k in range(3):
                    if j + k >= 2:
                        total += ca[j] * cL[2 - j] * cb[k] * cR[2 - k]
            s1 = S1 - a * b
            s2 = S2 - a * b * b
            s4 = S4 - a * a * b
            P = c2(Rp) - (SR - b * b - Rp) // 2
            Q = c2(Lp) - (SL - a * a - Lp) // 2
            total += a * (Lp * P - (Rp * s1 - s2)) + b * (Rp * Q - (Lp * s1 - s4))
            # x sang bên trái
            S1 -= l * r; S2 -= l * r * r; S4 -= l * l * r; SL -= l * l
            l += 1
            S1 += l * r; S2 += l * r * r; S4 += l * l * r; SL += l * l
            cl[x] = l
        return total % MOD

for a, exp in examples:
    got = Solution().subsequencesWithMiddleMode(a)
    print(got, 'OK' if got == exp else 'SAI')

6 OK
4 OK
0 OK


## 10. Vì sao thuật toán đúng

1. **Phân hoạch:** mỗi dãy con 5 phần tử có đúng một chỉ số giữa `i`; với `i` cố định, dãy con ↔ (cặp chỉ số trái, cặp chỉ số phải).
2. **Điều kiện (3.1):** hợp lệ ⇔ `t ≥ 3`, hoặc `t = 2` và ba phần tử còn lại đôi một khác giá trị (và khác `x` — tự động vì `t = 2`).
3. **(A)** đếm trực tiếp theo số `x` mỗi bên.
4. **(B)** với `j = 1, k = 0`: phần tử `x` bên trái (`a` cách) độc lập với phần còn lại; phần còn lại là `y` (bên trái, khác `x`) và cặp `{z1, z2}` (bên phải, khác `x`) với ba giá trị khác nhau. Với `y` có giá trị `v`, số cặp phải hợp lệ `= P − rc[v](R' − rc[v])` (mục 3.2) ⇒ tổng theo `y` ra công thức. **(C)** đối xứng.
5. **Cập nhật tổng:** khi dịch tâm, chỉ `cl[x]`, `cr[x]` thay đổi nên chỉ số hạng của `x` trong mỗi tổng thay đổi.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi bộ 5 chỉ số | `O(C(n, 5))` | `O(1)` |
| 2. Theo tâm, tính lại | `O(n · D)` (`D` số giá trị khác nhau) | `O(D)` |
| 3. Duy trì tổng | `O(n)` | `O(D)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'per_center': count_per_center, 'fast': count_fast, 'submit': lambda a: Solution().subsequencesWithMiddleMode(a)}
edge = [([7, 7, 7, 7, 7], 1), ([1, 2, 3, 4, 5], 0), ([1, 1, 2, 1, 1], 0), ([2, 1, 2, 3, 4], 1), ([1, 2, 1, 3, 4], 1), ([5, 6, 5, 6, 5, 7], None)]   # [1,1,2,1,1]: giữa là 2 chỉ xuất hiện 1 lần
for a, exp in examples + edge:
    b = count_brute(a)
    if exp is not None:
        assert b == exp, (a, b)
    for name, f in impls.items():
        assert f(a) == b, (name, a)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3416)
cnt = 0
for _ in range(1500):
    n = rng.randint(5, 11)
    a = [rng.randint(0, rng.randint(1, 4)) for _ in range(n)]
    exp = count_brute(a)
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
    cnt += 1
for _ in range(100):
    a = [rng.randint(0, 30) for _ in range(rng.randint(5, 300))]
    assert count_per_center(a) == count_fast(a) == Solution().subsequencesWithMiddleMode(a)
print('Khớp vét cạn trên', cnt, 'mảng nhỏ; cách 2, 3 và bài nộp khớp trên 100 mảng tới 300 phần tử.')

Khớp vét cạn trên 1500 mảng nhỏ; cách 2, 3 và bài nộp khớp trên 100 mảng tới 300 phần tử.


### 12.3. Ca lớn `n = 10^5`

In [12]:
import time
n = 10 ** 5
cases = {
    'giá trị khác nhau hết': [rng.randint(-10 ** 9, 10 ** 9) for _ in range(n)],
    'ít giá trị (1..50)': [rng.randint(1, 50) for _ in range(n)],
    'toàn bằng nhau': [7] * n,
    'hai giá trị xen kẽ': [i % 2 for i in range(n)],
}
for name, a in cases.items():
    t = time.time(); r = Solution().subsequencesWithMiddleMode(a); t1 = time.time() - t
    print(f"{name:22s}: {r:10d} | {t1:.2f}s")
assert cases and Solution().subsequencesWithMiddleMode([7] * n) == comb(n, 5) % MOD
print('toàn bằng nhau = C(n, 5) mod p: đúng')

giá trị khác nhau hết :  208252649 | 0.24s


ít giá trị (1..50)    :    9390230 | 0.23s
toàn bằng nhau        :  502061291 | 0.20s


hai giá trị xen kẽ    :  417687440 | 0.23s
toàn bằng nhau = C(n, 5) mod p: đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm trường hợp `t ≥ 3`

In [13]:
def wrong_only_ge3(nums):
    n, total = len(nums), 0
    for i in range(n):
        x = nums[i]
        L, R = Counter(nums[:i]), Counter(nums[i + 1:])
        a, b = L[x], R[x]; Lp, Rp = i - a, n - 1 - i - b
        total += sum(comb(a, j) * comb(Lp, 2 - j) * comb(b, k) * comb(Rp, 2 - k)
                     for j in range(3) for k in range(3) if j + k >= 2)
    return total % MOD

print('sai :', wrong_only_ge3([1, 2, 2, 3, 3, 4]), '| đúng:', Solution().subsequencesWithMiddleMode([1, 2, 2, 3, 3, 4]))

sai : 0 | đúng: 4


**Vì sao sai:** `x` xuất hiện **2 lần** vẫn có thể là mode duy nhất nếu ba phần tử còn lại khác nhau (ví dụ 2).

### Lỗi 2 — Trường hợp `t = 2` không yêu cầu ba phần tử còn lại khác nhau

In [14]:
def wrong_t2_any(nums):
    n, total = len(nums), 0
    for i in range(n):
        x = nums[i]
        L, R = Counter(nums[:i]), Counter(nums[i + 1:])
        a, b = L[x], R[x]; Lp, Rp = i - a, n - 1 - i - b
        total += sum(comb(a, j) * comb(Lp, 2 - j) * comb(b, k) * comb(Rp, 2 - k)
                     for j in range(3) for k in range(3) if j + k >= 1)   # SAI: t = 2 mọi trường hợp
    return total % MOD

print('sai :', wrong_t2_any([1, 2, 2, 3, 3, 4]), '| đúng:', Solution().subsequencesWithMiddleMode([1, 2, 2, 3, 3, 4]))

sai : 6 | đúng: 4


**Vì sao sai:** `[1, 2, 2, 3, 3]` có `2` và `3` cùng xuất hiện 2 lần — **hòa**, không phải mode duy nhất.

### Lỗi 3 — Dùng đếm **không loại `x`** khi tính các cặp khác giá trị

In [15]:
def wrong_include_x(nums):
    n, total = len(nums), 0
    for i in range(n):
        x = nums[i]
        L, R = Counter(nums[:i]), Counter(nums[i + 1:])
        a, b = L[x], R[x]; Lp, Rp = i - a, n - 1 - i - b
        total += sum(comb(a, j) * comb(Lp, 2 - j) * comb(b, k) * comb(Rp, 2 - k)
                     for j in range(3) for k in range(3) if j + k >= 2)
        lc, rc = dict(L), dict(R)                                   # SAI: còn chứa x
        P = C2(Rp) - sum(C2(c) for v, c in rc.items() if v != x)
        Q = C2(Lp) - sum(C2(c) for v, c in lc.items() if v != x)
        t1 = Lp * P - sum(lc[v] * rc.get(v, 0) * (Rp - rc.get(v, 0)) for v in lc)
        t2 = Rp * Q - sum(rc[v] * lc.get(v, 0) * (Lp - lc.get(v, 0)) for v in rc)
        total += a * t1 + b * t2
    return total % MOD

rng = random.Random(3)
ce = next(a for a in ([rng.randint(0, 3) for _ in range(rng.randint(5, 9))] for _ in range(20000)) if wrong_include_x(a) != count_brute(a))
print('phản ví dụ:', ce, '| sai:', wrong_include_x(ce), '| đúng:', count_brute(ce))

phản ví dụ: [1, 1, 3, 3, 3, 1, 1] | sai: 8 | đúng: 6


**Vì sao sai:** trong trường hợp `t = 2`, các phần tử `y, z` được chọn từ phần **khác `x`**; để số hạng của `x` trong tổng làm sai lệch kết quả.

### Lỗi 4 — Đếm cặp chỉ số **có thứ tự**

In [16]:
def wrong_ordered(nums):
    return count_per_center(nums) * 4 % MOD          # mỗi bên chọn 2 chỉ số có thứ tự: ×2×2

print('sai :', wrong_ordered([1, 1, 1, 1, 1, 1]), '| đúng:', Solution().subsequencesWithMiddleMode([1, 1, 1, 1, 1, 1]))

sai : 24 | đúng: 6


**Vì sao sai:** dãy con xác định bởi **tập** chỉ số; hai chỉ số bên trái phải theo thứ tự tăng ⇒ `C(·, 2)` chứ không phải `A(·, 2)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Dãy con độ dài cố định nhỏ ⇒ cố định **phần tử giữa** (hoặc một vị trí đặc biệt), đếm hai bên độc lập.
- Điều kiện "mode duy nhất" ⇒ chia theo số lần xuất hiện của mode.
- "Ba giá trị đôi một khác nhau" ⇒ tổng − cặp trùng, đưa về các tổng `Σ cl·cr^p`.
- Dịch tâm chỉ thay đổi một giá trị ⇒ cập nhật tổng `O(1)`.

## 15. Biến thể của bài

1. **LC 3395 — bản I:** cùng đề, `n ≤ 1000` (cách 2 là đủ).
2. Dãy con độ dài 3 với mode duy nhất ở giữa ⇒ chỉ cần `t ≥ 2`.
3. Đếm dãy con 5 phần tử có giá trị giữa là **trung vị** ⇒ đếm theo số phần tử nhỏ hơn/lớn hơn mỗi bên.
4. Mode duy nhất **bất kỳ vị trí** (không cần ở giữa) ⇒ cố định giá trị mode thay vì vị trí.

In [17]:
# Biến thể 2: dãy con độ dài 3 có mode duy nhất ở giữa = có ít nhất một phần tử bên cạnh bằng giá trị giữa
def count_len3(nums):
    n, total = len(nums), 0
    cl, cr = Counter(), Counter(nums)
    for i in range(n):
        x = nums[i]; cr[x] -= 1
        a, b = cl[x], cr[x]; L, R = i, n - 1 - i
        total += L * R - (L - a) * (R - b)                 # trừ trường hợp không bên nào bằng x
        cl[x] += 1
    return total

def brute_len3(nums):
    return sum(1 for idx in itertools.combinations(range(len(nums)), 3)
               if unique_middle_mode3([nums[i] for i in idx]))

def unique_middle_mode3(seq):
    c = Counter(seq); m = seq[1]
    return all(c[m] > v for k, v in c.items() if k != m)

rng = random.Random(15)
for _ in range(500):
    a = [rng.randint(0, 3) for _ in range(rng.randint(3, 10))]
    assert count_len3(a) == brute_len3(a)
print('Biến thể độ dài 3 khớp vét cạn:', count_len3([1, 2, 2, 3, 3, 4]))

Biến thể độ dài 3 khớp vét cạn: 8


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi bộ 5 chỉ số | `O(n^5)` | Oracle |
| 2 | Theo tâm, tính lại | `O(n · D)` | Đủ cho bản I |
| 3 | Duy trì tổng | `O(n)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[2, 2, 2, 2, 2, 2, 2]`.

**Bài 2 (tính tay):** `[1, 1, 2, 1, 1, 3]`.

**Bài 3 (điền TODO):** viết `pairs_distinct_avoiding(counts, v)` — số cặp khác giá trị không chứa `v`.

**Bài 4 (tự viết lại):** viết lại cách 3 với `dict` thường và lấy mod sau mỗi tâm (mô phỏng ngôn ngữ có tràn số).

**Bài 5 (mini challenge):** biến thể 4 — đếm dãy con 5 phần tử có **mode duy nhất** (ở bất kỳ vị trí nào).

In [18]:
def pairs_distinct_avoiding_todo(counts, v):
    # TODO
    raise NotImplementedError

def count_fast_mod_todo(nums):
    # TODO
    raise NotImplementedError

def count_unique_mode_anywhere_todo(nums):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mọi dãy con là `[2,2,2,2,2]`. (2) `t = 5`. (3) `C(7, 5) = 21`.

**Bài 2.** (1) Tâm là `1` với nhiều `1` hai bên. (2) Tâm `2` (chỉ số 2): `t = 1` ⇒ không. (3) Kiểm tra bằng vét cạn.

**Bài 3.** (1) `C(R, 2) − Σ C(c, 2)`. (2) Trừ `c_v (R − c_v)`. (3) Đối chiếu đếm trực tiếp.

**Bài 4.** (1) Các tổng có thể rất lớn. (2) Lấy mod sau mỗi phép nhân. (3) Cẩn thận phép chia 2 trong `(Σ r² − Σ r)/2` — dùng nghịch đảo của 2 hoặc tính `Σ C(r, 2)` trực tiếp.

**Bài 5.** (1) Cố định giá trị mode `v` và số lần `t`. (2) `t ≥ 3` luôn được; `t = 2` cần ba phần tử khác đôi một khác nhau. (3) Vét cạn với `n` nhỏ để kiểm tra.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
a1 = Solution().subsequencesWithMiddleMode([2] * 7)
a2 = Solution().subsequencesWithMiddleMode([1, 1, 2, 1, 1, 3])
assert a1 == comb(7, 5) == 21 and a2 == count_brute([1, 1, 2, 1, 1, 3])
print('Bài 1:', a1, '| Bài 2:', a2)

def pairs_distinct_avoiding(counts, v):
    R = sum(counts.values())
    P = C2(R) - sum(C2(c) for c in counts.values())
    return P - counts.get(v, 0) * (R - counts.get(v, 0))

vals = [1, 1, 2, 3, 3, 3, 4]
cnt_ = Counter(vals)
for v in [1, 2, 3, 4, 9]:
    exp = sum(1 for i, j in itertools.combinations(range(len(vals)), 2) if vals[i] != vals[j] and v not in (vals[i], vals[j]))
    assert pairs_distinct_avoiding(cnt_, v) == exp
print('Bài 3 khớp đếm trực tiếp.')

Bài 1: 21 | Bài 2: 3
Bài 3 khớp đếm trực tiếp.


In [20]:
# Bài 4, 5
INV2 = pow(2, MOD - 2, MOD)

def count_fast_mod(nums):
    n = len(nums)
    cl, cr = {}, {}
    for v in nums:
        cr[v] = cr.get(v, 0) + 1
    S1 = S2 = S4 = SL = 0
    SR = sum(c * c for c in cr.values()) % MOD
    total = 0
    c2 = lambda x: x * (x - 1) % MOD * INV2 % MOD
    for i in range(n):
        x = nums[i]; l, r = cl.get(x, 0), cr[x]
        S1 = (S1 - l * r) % MOD; S2 = (S2 - l * r * r) % MOD; S4 = (S4 - l * l * r) % MOD; SR = (SR - r * r) % MOD
        r -= 1; cr[x] = r
        S1 = (S1 + l * r) % MOD; S2 = (S2 + l * r * r) % MOD; S4 = (S4 + l * l * r) % MOD; SR = (SR + r * r) % MOD
        a, b = l, r; Lp, Rp = i - a, n - 1 - i - b
        ca, cb, cL, cR = (1, a, c2(a)), (1, b, c2(b)), (1, Lp, c2(Lp)), (1, Rp, c2(Rp))
        for j in range(3):
            for k in range(3):
                if j + k >= 2:
                    total = (total + ca[j] * cL[2 - j] % MOD * cb[k] % MOD * cR[2 - k]) % MOD
        s1 = (S1 - a * b) % MOD; s2 = (S2 - a * b * b) % MOD; s4 = (S4 - a * a * b) % MOD
        P = (c2(Rp) - (SR - b * b - Rp) * INV2) % MOD
        Q = (c2(Lp) - (SL - a * a - Lp) * INV2) % MOD
        total = (total + a * (Lp * P - (Rp * s1 - s2)) + b * (Rp * Q - (Lp * s1 - s4))) % MOD
        S1 = (S1 - l * r) % MOD; S2 = (S2 - l * r * r) % MOD; S4 = (S4 - l * l * r) % MOD; SL = (SL - l * l) % MOD
        l += 1; cl[x] = l
        S1 = (S1 + l * r) % MOD; S2 = (S2 + l * r * r) % MOD; S4 = (S4 + l * l * r) % MOD; SL = (SL + l * l) % MOD
    return total

def count_unique_mode_anywhere_brute(nums):
    def ok(seq):
        c = sorted(Counter(seq).values(), reverse=True)
        return len(c) == 1 or c[0] > c[1]
    return sum(1 for idx in itertools.combinations(range(len(nums)), 5) if ok([nums[i] for i in idx]))

rng = random.Random(19)
for _ in range(300):
    a = [rng.randint(0, 4) for _ in range(rng.randint(5, 12))]
    assert count_fast_mod(a) == count_brute(a)
big = [rng.randint(1, 40) for _ in range(5000)]
assert count_fast_mod(big) == Solution().subsequencesWithMiddleMode(big)
print('Bài 4 khớp (kể cả n = 5000). Bài 5 (vét cạn) với [1,1,2,2,3,3,1]:', count_unique_mode_anywhere_brute([1, 1, 2, 2, 3, 3, 1]))

Bài 4 khớp (kể cả n = 5000). Bài 5 (vét cạn) với [1,1,2,2,3,3,1]: 6


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cho mỗi tâm i (x = nums[i]): a, b = số x trái/phải; L', R' = số khác x
(A) Σ_{j+k>=2} C(a,j)C(L',2-j)C(b,k)C(R',2-k)
(B) a·(L'·P − (R'·s1 − s2)),  P = C(R',2) − Σ_{v≠x} C(rc,2)
(C) b·(R'·Q − (L'·s1 − s4)),  Q = C(L',2) − Σ_{v≠x} C(lc,2)
s1 = Σ_{v≠x} lc·rc, s2 = Σ_{v≠x} lc·rc², s4 = Σ_{v≠x} lc²·rc   (tổng toàn cục − số hạng của x)
```

**Checklist:**

- [ ] Có cả trường hợp `t = 2` (ba phần tử còn lại đôi một khác nhau)?
- [ ] Loại giá trị `x` khỏi các tổng ở trường hợp `t = 2`?
- [ ] Chọn chỉ số bằng tổ hợp `C(·, 2)`?
- [ ] Cập nhật tổng khi `x` rời phải và khi sang trái?
- [ ] Đối chiếu chữ ký hàm nếu có Premium?